In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1. Let's load the WikiHow 2 Dataset using Hugging Face

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    return tokenizer(example["text"])

tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

tokenized_dataset.save_to_disk("/content/drive/MyDrive/wikitext_tokenized")

In [ ]:
dataset['train'][1]['text']

' = Valkyria Chronicles III = \n'

In [ ]:
tokenized_dataset['train'][1]

{'input_ids': [796, 569, 18354, 7496, 17740, 6711, 796, 220, 198],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
from datasets import load_from_disk

# 🔁 Load instead of recomputing after restart
tokenized_dataset = load_from_disk("/content/drive/MyDrive/wikitext_tokenized")

def group_texts(examples, block_size):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])

    total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i:i+block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }

    result["labels"] = result["input_ids"].copy()
    return result

# 🔥 Precompute datasets for different context lengths
lm_datasets = {}

for L in [512, 1024, 2048, 3072]:  # add 2048 later if needed
    print(f"Processing context length {L}...")

    lm_dataset = tokenized_dataset.map(
        lambda x: group_texts(x, block_size=L),
        batched=True
    )

    lm_dataset.set_format("torch")

    lm_dataset.save_to_disk(f"/content/drive/MyDrive/wikitext_{L}")

    lm_datasets[L] = lm_dataset

In [ ]:
vocab_size = len(tokenizer)
vocab_size

50257

In [ ]:
from datasets import load_from_disk
from torch.utils.data import DataLoader

# 🔁 Load datasets after restart
lm_datasets = {
    512: load_from_disk("/content/drive/MyDrive/wikitext_512"),
    1024: load_from_disk("/content/drive/MyDrive/wikitext_1024"),
    2048: load_from_disk("/content/drive/MyDrive/wikitext_2048"),
    3072: load_from_disk("/content/drive/MyDrive/wikitext_3072")
}

# IMPORTANT: set format after loading
for L in lm_datasets:
    lm_datasets[L].set_format("torch")

def get_dataloaders(seq_len, batch_size):
    dataset = lm_datasets[seq_len]

    train_loader = DataLoader(dataset["train"], shuffle=True, batch_size=batch_size)
    val_loader = DataLoader(dataset["validation"], shuffle=False, batch_size=batch_size)
    test_loader = DataLoader(dataset["test"], shuffle=False, batch_size=batch_size)

    return train_loader, val_loader, test_loader

# 2. Let's create a Basic Transformer Model

First, we create the Masked self attention and MLP blocks.

In [ ]:
import torch
from torch import nn
import math
import torch.nn.functional as F

# device agnostic code
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class MaskedSelfAttention(nn.Module):

  def __init__(self,config):

    super().__init__()
    assert config['d_model'] % config['num_heads'] == 0, f"Embedding dimension must be divisible by number of heads"

    self.embedding_dim = config['d_model']
    self.num_heads = config['num_heads']
    self.d_k = config['d_model'] // config['num_heads']

    # a single linear layer to project Q,K and V all once
    self.proj_attn = nn.Linear(config['d_model'],3*config['d_model'])

    # output projection
    self.output_proj = nn.Linear(config['d_model'],config['d_model'])

    # Dropout layers
    self.attn_dropout = nn.Dropout(config['dropout'])
    self.residual_dropout = nn.Dropout(config['dropout'])

    if config['pos_type'] == 'rope':
      self.rope = RoPE(config)
    else:
      self.rope = None

    if config['pos_type'] == 'alibi':
      self.alibi = ALiBi(config)
    else:
      self.alibi = None

    if config['pos_type'] == 'relative':
      self.pos_embedding = RelativePositionalEncoding(config)
    else:
      self.pos_embedding = None

    mask = torch.tril(torch.ones(config['seq_len'],config['seq_len']))
    mask = mask.view(1,1,config['seq_len'],config['seq_len'])

    # Since we don't want the mask to be a learnable parameter, make it into a buffer
    self.register_buffer('bias',mask)

  def forward(self,x):

    batch_size,seq_len,_ = x.shape

    # split into query, key and value
    qkv = self.proj_attn(x)
    q,k,v = qkv.split(self.embedding_dim,dim=2)

    # split up the embedding dimension over 8 heads to get d_k = 64 for each, and then a transpose so that all the data
    # for a single head is grouped together
    q = q.view(batch_size,seq_len,self.num_heads,self.d_k).transpose(1,2)
    k = k.view(batch_size,seq_len,self.num_heads,self.d_k).transpose(1,2)
    v = v.view(batch_size,seq_len,self.num_heads,self.d_k).transpose(1,2)

    if self.rope is not None:

      position_ids = torch.arange(seq_len,device=x.device)
      position_ids = position_ids.unsqueeze(0).expand(batch_size,-1)

      q,k = self.rope(q,k,position_ids)

    if self.pos_embedding is not None:
      rel_embeddings = self.pos_embedding(seq_len)

      rel_scores = torch.einsum('bhid,ijd->bhij',q,rel_embeddings)

      scores = (q @ k.transpose(-2,-1) + rel_scores) / math.sqrt(self.d_k)

    else:

      # now the shape is (batch_size,num_heads,seq_len,d_k)
      # we perform the self attention operation now
      # (batch_size,num_heads,seq_len,d_k) @ (batch_size,num_heads,d_k,seq_len) = (batch_len,num_heads,seq_len,seq_len)
      scores = q @ k.transpose(-2,-1) / math.sqrt(self.d_k)

    if self.alibi is not None:

      scores += self.alibi(seq_len)

    # Expand causal mask if extrapolating beyond training length
    if seq_len > self.bias.shape[-1]:
        mask = torch.tril(
        torch.ones(
            seq_len,
            seq_len,
            device=x.device
        )
    ).view(1,1,seq_len,seq_len)

    else:
      mask = self.bias[:,:,:seq_len,:seq_len]

    # applying the mask, replacing 0 with -inf for softmax later
    scores = scores.masked_fill(mask == 0,-1e4)

    # softmax and dropout
    weights = F.softmax(scores,dim=-1)
    weights = self.attn_dropout(weights)

    # Multiply by V
    out = weights @ v

    # reassemble the heads
    out = out.transpose(1,2).contiguous().view(batch_size,seq_len,self.embedding_dim)

    # final projection
    return self.residual_dropout(self.output_proj(out))

In [ ]:
from torch import nn

class MLPBlock(nn.Module):

  def __init__(self,config):

    super().__init__()

    self.mlp = nn.Sequential(
        nn.Linear(config['d_model'],config['d_model']*4),
        nn.ReLU(),
        nn.Dropout(config['dropout']),
        nn.Linear(config['d_model']*4,config['d_model']),
        nn.Dropout(config['dropout'])
    )

  def forward(self,x):

    return self.mlp(x)

Let's combine these two together to create the Transformer Layer, with the Post layer norm approach.

In [ ]:
from torch import nn

class TransformerLayer(nn.Module):

  def __init__(self,config):

    super().__init__()

    self.use_conv = config.get('interleaved_conv',False)

    if self.use_conv:
      self.conv = CausalConv1D(config)
      self.ln_conv = nn.LayerNorm(config['d_model'])

    self.attn = get_attention(config['attn_type'],config)

    if config.get('gated_ffn',False):
      self.mlp = GatedConvFFN(config)
    else:
      self.mlp = MLPBlock(config)

    self.layer_norm1 = nn.LayerNorm(config['d_model'])
    self.layer_norm2 = nn.LayerNorm(config['d_model'])
    self.dropout = nn.Dropout(config['dropout'])

  def forward(self,x):

    if self.use_conv:
      x = self.ln_conv(self.dropout(self.conv(x)) + x)

    x = self.layer_norm1(self.dropout(self.attn(x)) + x) # Residual connection
    x = self.layer_norm2(self.dropout(self.mlp(x)) + x)
    return x

Now, we create the sinusoidal positional embeddings.

In [ ]:
import torch
from torch import nn
import math

class PositionalEncoding(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.d_model = config['d_model']

        self.register_buffer(
            "pe",
            self._build_pe(config['seq_len'])
        )

    def _build_pe(self, seq_len):

        pe = torch.zeros(seq_len, self.d_model)

        position = torch.arange(
            0, seq_len,
            dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0, self.d_model, 2
            ).float()
            * (-math.log(10000.0) / self.d_model)
        )

        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)

        return pe.unsqueeze(0)

    def forward(self, x):

        seq_len = x.shape[1]

        if seq_len > self.pe.shape[1]:
            self.pe = self._build_pe(seq_len).to(x.device)

        return x + self.pe[:, :seq_len, :]

Finally, we combine everything to create the entire transformer.

In [ ]:
from torch import nn
import math

class TransformerModel(nn.Module):

  def __init__(self,config):

    super().__init__()

    self.embedding_dim = config['d_model']

    self.embedding = nn.Embedding(config['vocab_size'],config['d_model'])

    if config['pos_type'] == 'sin':
      self.pos_embedding = PositionalEncoding(config)
    else:
      self.pos_embedding = None

    self.embedding_dropout = nn.Dropout(config['dropout'])

    self.transformer_blocks = nn.ModuleList(
        TransformerLayer(config)
        for _ in range(config['n_layers'])
    )

    self.output_layer = nn.Linear(config['d_model'],config['vocab_size'],bias=False)

    # using the same weight matrix
    self.output_layer.weight = self.embedding.weight

    self.apply(self._init_weights)

  def _init_weights(self,module):
    if isinstance(module,nn.Linear):
      torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module,nn.Embedding):
      torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)

  def forward(self,x):

    # We scale the embeddings as well to ensure they don't get drowned out by the positional embeddings
    x = self.embedding(x) * math.sqrt(self.embedding_dim)

    if self.pos_embedding is not None:
      x = self.pos_embedding(x)

    x = self.embedding_dropout(x)

    for block in self.transformer_blocks:
      x = block(x)

    x = self.output_layer(x)

    return x

# 3. Training Loop

In [ ]:
import torch
import torch.nn.functional as F
import time
import math

def train(model, train_dataloader, val_dataloader, optimizer, scheduler, loss_fn, epochs=1, device="cuda", accumulation_steps=4,max_steps=870):
    model.to(device)

    scaler = torch.cuda.amp.GradScaler()

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_perplexity": [],
        "throughput": [],
        "peak_memory": [],
        "epoch_time": []
    }

    best_val_loss = float('inf')
    global_step = 0

    for epoch in range(epochs):
        print(f"--- Epoch {epoch+1}/{epochs} ---")

        # ================= TRAIN =================
        model.train()
        torch.cuda.reset_peak_memory_stats()

        total_train_loss = 0
        total_tokens = 0
        total_tokens_processed = 0

        start_time = time.time()
        optimizer.zero_grad()

        for step, batch in enumerate(train_dataloader):
            global_step += 1
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            inputs = input_ids[:, :-1]
            targets = labels[:, 1:]

            # Autocast to FP16 (or consider 'bfloat16' if supported and you see NaN losses)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(inputs)
                loss = loss_fn(
                    logits.reshape(-1, logits.size(-1)),
                    targets.reshape(-1)
                )
                # Scale loss down for gradient accumulation
                loss = loss / accumulation_steps

            scaler.scale(loss).backward()

            # Check if it's time to step OR if it's the last batch in the dataloader
            is_accumulation_boundary = (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_dataloader)

            if is_accumulation_boundary:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                # Store the scale to check if a step is skipped
                scale_before = scaler.get_scale()

                scaler.step(optimizer)
                scaler.update()

                scale_after = scaler.get_scale()

                # ONLY step the scheduler if the optimizer actually applied the gradients
                if scale_before <= scale_after:
                    scheduler.step()

                optimizer.zero_grad()

                if max_steps is not None and global_step >= max_steps:
                  break

            # Token-level accounting
            # Note: If using padding, replace targets.numel() with (targets != -100).sum().item()
            batch_tokens = (targets != -100).sum().item()
            total_train_loss += loss.item() * accumulation_steps * batch_tokens
            total_tokens += batch_tokens
            total_tokens_processed += inputs.numel()

            if step % 50 == 0:
                print(f"  Step {step} | Loss: {loss.item() * accumulation_steps:.4f}")

        epoch_time = time.time() - start_time
        throughput = total_tokens_processed / epoch_time

        avg_train_loss = total_train_loss / total_tokens
        peak_mem = torch.cuda.max_memory_allocated() / 1e9  # GB

        print(f"Train Loss: {(loss.item() * accumulation_steps):.4f} | Throughput: {throughput:.2f} tokens/sec | Peak Mem: {peak_mem:.2f} GB | epoch time: {epoch_time:.2f} sec")

        # ================= VALIDATION =================
        model.eval()

        total_val_loss = 0
        total_val_tokens = 0

        with torch.inference_mode():
            for batch in val_dataloader:
                input_ids = batch["input_ids"].to(device)
                labels = batch["labels"].to(device)

                inputs = input_ids[:, :-1]
                targets = labels[:, 1:]

                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(inputs)
                    loss = loss_fn(
                        logits.reshape(-1, logits.size(-1)),
                        targets.reshape(-1)
                    )

                batch_tokens = targets.numel() # Again, watch out for padding tokens here
                total_val_loss += loss.item() * batch_tokens
                total_val_tokens += batch_tokens

        avg_val_loss = total_val_loss / total_val_tokens
        val_perplexity = math.exp(min(avg_val_loss, 25))

        print(f"Val Loss: {avg_val_loss:.4f} | Val Perplexity: {val_perplexity:.2f}\n")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            print(f"New best validation loss: {best_val_loss:.4f}! Saving model...")
            torch.save(model.state_dict(), "best_baseline_transformer.pth")

        # ================= LOGGING =================
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_perplexity"].append(val_perplexity)
        history["throughput"].append(throughput)
        history["peak_memory"].append(peak_mem)
        history["epoch_time"].append(epoch_time)

        if max_steps is not None and global_step >= max_steps:
          break

    return history

# 4. Attention Variants

We experiment with multiple attention variants such as Local attention, Multi-Query Attention and Linear Attention.

## 4.1 Multi/Group Query Attention

In [ ]:
import torch
from torch import nn
import math
import torch.nn.functional as F

# device agnostic code
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class GQA(nn.Module):

  def __init__(self,config):

    super().__init__()
    assert config['d_model'] % config['num_heads'] == 0, f"Embedding dimension must be divisible by number of heads"
    assert config['num_heads'] % config.get('groups',1) == 0, f"number of heads must be divisible by number of groups"

    self.embedding_dim = config['d_model']
    self.num_heads = config['num_heads']
    self.d_k = config['d_model'] // config['num_heads']
    self.heads_per_group = config['num_heads'] // config.get('groups',1)
    self.k_v_dim = config.get('groups',1) * self.d_k
    self.groups = config.get('groups',1)

    # 3 diff layers to project all 3 differently
    self.q_attn = nn.Linear(config['d_model'],config['d_model'])
    self.k_attn = nn.Linear(config['d_model'],self.k_v_dim)
    self.v_attn = nn.Linear(config['d_model'],self.k_v_dim)

    # output projection
    self.output_proj = nn.Linear(config['d_model'],config['d_model'])

    # Dropout layers
    self.attn_dropout = nn.Dropout(config['dropout'])
    self.residual_dropout = nn.Dropout(config['dropout'])

  def forward(self,x,kv_cache = None):

    batch_size,seq_len,_ = x.shape

    # split into query, key and value
    q = self.q_attn(x)
    k = self.k_attn(x)
    v = self.v_attn(x)

    q = q.view(batch_size,seq_len,self.num_heads,self.d_k).transpose(1,2)
    k = k.view(batch_size,seq_len,self.groups,self.d_k).transpose(1,2)
    v = v.view(batch_size,seq_len,self.groups,self.d_k).transpose(1,2)

    if kv_cache is not None:
      k_prev,v_prev = kv_cache
      k = torch.cat([k_prev,k],dim=2)
      v = torch.cat([v_prev,v],dim=2)

    new_kv_cache = (k,v)

    k = k.unsqueeze(2).expand(-1, -1, self.heads_per_group, -1, -1)
    k = k.reshape(batch_size, self.num_heads, -1, self.d_k)
    v = v.unsqueeze(2).expand(-1, -1, self.heads_per_group, -1, -1)
    v = v.reshape(batch_size, self.num_heads, -1, self.d_k)

    # now the shape is (batch_size,num_heads,seq_len,d_k)
    # we perform the self attention operation now
    # (batch_size,num_heads,seq_len,d_k) @ (batch_size,num_heads,d_k,seq_len) = (batch_len,num_heads,seq_len,seq_len)
    scores = q @ k.transpose(-2,-1) / math.sqrt(self.d_k)

    # no need during inference
    if seq_len > 1 or kv_cache is None:
      T_k = k.size(2)
      mask = torch.tril(torch.ones(seq_len,T_k,device = x.device)).bool()
      mask = mask.view(1,1,seq_len,T_k)

      # applying the mask, replacing 0 with -inf for softmax later
      scores = scores.masked_fill(~mask,-1e4)

    # softmax and dropout
    weights = F.softmax(scores,dim=-1)
    weights = self.attn_dropout(weights)

    # Multiply by V
    out = weights @ v

    # reassemble the heads
    out = out.transpose(1,2).contiguous().view(batch_size,seq_len,self.embedding_dim)

    # final projection
    return self.residual_dropout(self.output_proj(out)) # not returning kv cache for now

## 4.2 Sliding Window/Local Attention

In [ ]:
import torch
from torch import nn
import math
import torch.nn.functional as F

# device agnostic code
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class SlidingWindow(nn.Module):

  def __init__(self,config):

    super().__init__()
    assert config['d_model'] % config['num_heads'] == 0, f"Embedding dimension must be divisible by number of heads"

    self.embedding_dim = config['d_model']
    self.num_heads = config['num_heads']
    self.d_k = config['d_model'] // config['num_heads']
    self.window = config.get('window',128)

    self.attn = nn.Linear(config['d_model'],3*config['d_model'])

    # output projection
    self.output_proj = nn.Linear(config['d_model'],config['d_model'])

    # Dropout layers
    self.attn_dropout = nn.Dropout(config['dropout'])
    self.residual_dropout = nn.Dropout(config['dropout'])

  def forward(self,x,kv_cache = None):

    batch_size,seq_len,_ = x.shape

    # split into query, key and value
    qkv = self.attn(x)
    q,k,v = qkv.split(self.embedding_dim,dim=2)

    q = q.view(batch_size,seq_len,self.num_heads,self.d_k).transpose(1,2)
    k = k.view(batch_size,seq_len,self.num_heads,self.d_k).transpose(1,2)
    v = v.view(batch_size,seq_len,self.num_heads,self.d_k).transpose(1,2)

    if kv_cache is not None:
      k_prev,v_prev = kv_cache
      curr = k_prev.size(2)
      if curr > self.window:
        k_prev = k_prev[:,:,curr-self.window:,:]
        v_prev = v_prev[:,:,curr-self.window:,:]
      k = torch.cat([k_prev,k],dim=2)
      v = torch.cat([v_prev,v],dim=2)

    new_kv_cache = (k,v)

    # now the shape is (batch_size,num_heads,seq_len,d_k)
    # we perform the self attention operation now
    # (batch_size,num_heads,seq_len,d_k) @ (batch_size,num_heads,d_k,seq_len) = (batch_len,num_heads,seq_len,seq_len)
    scores = q @ k.transpose(-2,-1) / math.sqrt(self.d_k)

    # no need during inference
    if seq_len > 1 or kv_cache is None:
      T_k = k.size(2)
      mask = torch.tril(torch.ones(seq_len,T_k,device = x.device)).bool()
      mask = mask.view(1,1,seq_len,T_k)

      sliding_window_mask = torch.tril(input=torch.ones(seq_len,T_k,device = x.device),diagonal=-self.window).bool()
      sliding_window_mask = sliding_window_mask.view(1,1,seq_len,T_k)

      # applying the mask, replacing 0 with -inf for softmax later
      scores = scores.masked_fill(~mask | sliding_window_mask,-1e4)

    # softmax and dropout
    weights = F.softmax(scores,dim=-1)
    weights = self.attn_dropout(weights)

    # Multiply by V
    out = weights @ v

    # reassemble the heads
    out = out.transpose(1,2).contiguous().view(batch_size,seq_len,self.embedding_dim)

    # final projection
    return self.residual_dropout(self.output_proj(out)) # not returning kv cache for now

## 4.3 Linear Attention

In [ ]:
import torch
from torch import nn
import math
import torch.nn.functional as F

# device agnostic code
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def phi(x):
  return 1.0 + F.elu(x)

class LinearAttention(nn.Module):

  def __init__(self,config):

    super().__init__()
    assert config['d_model'] % config['num_heads'] == 0, f"Embedding dimension must be divisible by number of heads"

    self.embedding_dim = config['d_model']
    self.num_heads = config['num_heads']
    self.d_k = config['d_model'] // config['num_heads']

    self.attn = nn.Linear(config['d_model'],3*config['d_model'])

    # output projection
    self.output_proj = nn.Linear(config['d_model'],config['d_model'])

    # Dropout layers
    self.attn_dropout = nn.Dropout(config['dropout'])
    self.residual_dropout = nn.Dropout(config['dropout'])

  def forward(self,x):

    batch_size,seq_len,_ = x.shape

    # split into query, key and value
    qkv = self.attn(x)
    q,k,v = qkv.split(self.embedding_dim,dim=2)

    q = q.view(batch_size,seq_len,self.num_heads,self.d_k).transpose(1,2).float()
    k = k.view(batch_size,seq_len,self.num_heads,self.d_k).transpose(1,2).float()
    v = v.view(batch_size,seq_len,self.num_heads,self.d_k).transpose(1,2).float()

    phi_k = phi(k)
    phi_q = phi(q)

    phi_q = phi_q / (phi_q.sum(dim=-1, keepdim=True) + 1e-6)
    phi_k = phi_k / (phi_k.sum(dim=-1, keepdim=True) + 1e-6)

    # creating the context matrix
    context_matrix = phi_k.unsqueeze(-1) @ v.unsqueeze(-2)

    # building the causual history
    s = torch.cumsum(context_matrix,dim=2)
    z = torch.cumsum(phi_k,dim=2)

    eps = 1e-4

    numerator = (phi_q.unsqueeze(-2) @ s).squeeze(-2)
    denominator = (phi_q * z).sum(dim=-1,keepdim=True)
    denominator = torch.clamp(denominator,min=eps)

    out = numerator / denominator

    # reassemble the heads
    out = out.transpose(1,2).contiguous().view(batch_size,seq_len,self.embedding_dim)

    # final projection
    return self.residual_dropout(self.output_proj(out))

# 5. Positional Encoding Variants

We implement modern positional encoding variants such as RoPE, ALiBi and relative positional encoding.

## 5.1 RoPE

In [ ]:
import torch
from torch import nn
import math

def rotate_half(x):
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)

class RoPE(nn.Module):

    def __init__(self,config):

        super().__init__()

        self.embedding_dim = config['d_model'] // config['num_heads']
        self.max_seq_len = config['seq_len']

        idxs = torch.arange(0, self.embedding_dim // 2)
        theta = config['base'] ** ((-2 * idxs) / self.embedding_dim)

        # register as buffer so it moves with device
        self.register_buffer("theta", theta)

        # precompute cache
        self._build_cache(config['seq_len'],self.theta.device)

    def _build_cache(self, seq_len,device):
        positions = torch.arange(seq_len,device=device).unsqueeze(1)  # (seq_len, 1)

        angles = positions * self.theta.unsqueeze(0)  # (seq_len, dim/2)

        angles = angles.unsqueeze(-1).expand(-1, -1, 2).reshape(seq_len, -1)

        sin = torch.sin(angles)
        cos = torch.cos(angles)

        self.register_buffer("sin_cache", sin)
        self.register_buffer("cos_cache", cos)

    def forward(self, q, k, position_ids):

        # rebuild cache during inference if too small
        seq_len = position_ids.max().item() + 1
        if seq_len > self.sin_cache.shape[0]:
            self._build_cache(seq_len,q.device)

        sin = self.sin_cache[position_ids].unsqueeze(1)
        cos = self.cos_cache[position_ids].unsqueeze(1)

        q_rotated = rotate_half(q) * sin + q * cos
        k_rotated = rotate_half(k) * sin + k * cos

        return q_rotated, k_rotated

## 5.2 ALiBi

In [ ]:
import torch
from torch import nn

class ALiBi(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.num_heads = config['num_heads']

        ratio = 2 ** (8.0 / self.num_heads)
        slopes = 1.0 / ratio ** torch.arange(
            1, self.num_heads + 1
        )

        self.register_buffer("slopes", slopes)

        self.register_buffer(
            "alibi_bias",
            self._build_bias(config['seq_len'])
        )

    def _build_bias(self, seq_len):

        device = self.slopes.device

        i = torch.arange(
            seq_len,
            device=device
        ).unsqueeze(1)

        j = torch.arange(
            seq_len,
            device=device
        ).unsqueeze(0)

        distances = j - i

        bias = (
            distances
            * self.slopes.unsqueeze(1).unsqueeze(2)
        )

        return bias

    def forward(self, seq_len):

        if seq_len > self.alibi_bias.shape[-1]:
            self.alibi_bias = self._build_bias(seq_len)

        return self.alibi_bias[:, :seq_len, :seq_len]

## 5.3 Relative Positional Encoding

In [ ]:
import torch
from torch import nn

class RelativePositionalEncoding(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.max_distance = config['max_distance']

        num_embeddings = (
            2 * self.max_distance + 1
        )

        self.embeddings = nn.Embedding(
            num_embeddings,
            config['head_dim']
        )

    def forward(self, seq_len: int):

        device = self.embeddings.weight.device

        i = torch.arange(
            seq_len,
            device=device
        ).unsqueeze(1)

        j = torch.arange(
            seq_len,
            device=device
        ).unsqueeze(0)

        distances = i - j

        distances = torch.clamp(
            distances,
            min=-self.max_distance,
            max=self.max_distance
        )

        distances += self.max_distance

        rel_embeddings = self.embeddings(distances)

        return rel_embeddings

#6. Convolution + Attention

First, we construct a Causal Conv1D block which will act as the building block for the next implementations.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class CausalConv1D(nn.Module):

  def __init__(self,config):
    super().__init__()

    self.pad = config['kernel_size'] - 1

    self.conv = nn.Conv1d(in_channels=config['d_model'],out_channels=config['d_model'],kernel_size=config['kernel_size'],groups=config['d_model'])

  def forward(self,x):

    x = x.transpose(1,2)

    x = F.pad(x, (self.pad,0))

    return self.conv(x).transpose(1,2)

## Gated Convolutional Feed Forward Networks

In [ ]:
from torch import nn

class GatedConvFFN(nn.Module):

  def __init__(self,config):
    super().__init__()

    # Information path
    self.linearInfo = nn.Linear(config['d_model'],config['d_model'] * 4)

    # Gated path
    self.linearGate = nn.Linear(config['d_model'],config['d_model'] * 4)
    conv_config = config.copy()
    conv_config['d_model'] = config['d_model'] * 4
    self.convGate = CausalConv1D(conv_config)
    self.activation = nn.GELU()

    # output
    self.linearOut = nn.Linear(config['d_model'] * 4,config['d_model'])

  def forward(self,x):

    info = self.linearInfo(x)

    gated = self.activation(self.convGate(self.linearGate(x)))

    gated_info = gated * info

    return self.linearOut(gated_info)

# 7. Experiments

In [ ]:
attention_variants = [
    {"attn": "normal"},
    {"attn": "gqa"},
    {"attn": "linear"},
    {"attn": "sliding"},
]

pos_variants = [
    {"pos" : "sin"},
    {"pos" : "rope"},
    {"pos" : "alibi"},
    {"pos" : "relative"},
]

# Define your 3 specific variants
experiment_variants = [
    {"interleaved": False, "gated": False},
    {"interleaved": True, "gated": False},
    {"interleaved": False, "gated": True},
]

context_lengths = [1024]
results = []

In [ ]:
def get_attention(attn_type, config):
    if attn_type == "normal":
        return MaskedSelfAttention(config)
    elif attn_type == "gqa":
        return GQA(config)
    elif attn_type == "linear":
        return LinearAttention(config)
    elif attn_type == "sliding":
        return SlidingWindow(config)
    else:
        raise ValueError(f"Unknown attention type: {attn_type}")

In [ ]:
def build_model(config):
    default_config = {
        "vocab_size": 50257,
        "d_model": 512,
        "num_heads": 8,
        "n_layers": 6,
        "dropout": 0.1,
        "window": 512,
        "groups": 2,
        "base": 10000,
        "max_distance": 128,
        "head_dim": 64,
        "kernel_size" : 3,
    }

    default_config.update(config)

    return TransformerModel(default_config)

## Visualizing the model

In [ ]:
import torch
try:
  from torchinfo import summary
except:
  !pip install -q torchinfo
  from torchinfo import summary

config = {
            "attn_type": "normal",
            "pos_type": "sin",
            "seq_len": 1024,
        }

model = build_model(config)

true_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"True Parameter Count: {true_params:,}")

print(summary(model,input_size=(8,1024),dtypes=[torch.long],col_names=['input_size','output_size','num_params'],row_settings=['var_names']))

del model
torch.cuda.empty_cache()

True Parameter Count: 44,645,888
Layer (type (var_name))                            Input Shape               Output Shape              Param #
TransformerModel (TransformerModel)                [8, 1024]                 [8, 1024, 50257]          --
├─Embedding (embedding)                            [8, 1024]                 [8, 1024, 512]            25,731,584
├─PositionalEncoding (pos_embedding)               [8, 1024, 512]            [8, 1024, 512]            --
├─Dropout (embedding_dropout)                      [8, 1024, 512]            [8, 1024, 512]            --
├─ModuleList (transformer_blocks)                  --                        --                        --
│    └─TransformerLayer (0)                        [8, 1024, 512]            [8, 1024, 512]            --
│    │    └─MaskedSelfAttention (attn)             [8, 1024, 512]            [8, 1024, 512]            1,050,624
│    │    └─Dropout (dropout)                      [8, 1024, 512]            [8, 1024, 512]        

Looks like torchinfo is counting the 25M params from the vocab_size to embedding_dim layers twice, even though I have specified that the weights are shared. Interesting!

## Run Experiments!

In [ ]:
import json

def save_results(results, path="/content/drive/MyDrive/exp_results.json"):
    with open(path, "w") as f:
        json.dump(results, f, indent=4)

def load_results(path="/content/drive/MyDrive/exp_results.json"):
    try:
        with open(path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        return []

## Main experiment runner

In [ ]:
from transformers import get_cosine_schedule_with_warmup
import torch

torch.manual_seed(42)

base_tokens = 8192
epochs = 3

for ctx_len in context_lengths:
    print(f"\n==============================")
    print(f"Context Length: {ctx_len}")
    print(f"==============================\n")

    # 🔴 rebuild dataloaders for this context length
    batch_size = base_tokens // ctx_len
    train_loader, val_loader,_ = get_dataloaders(seq_len=ctx_len,batch_size=batch_size)

    for attn in attention_variants:
      for pos in pos_variants:
        for exp in experiment_variants:

          print(f"\n--- Running: {attn['attn']} | ctx={ctx_len} | pos={pos['pos']} | interleaved={exp['interleaved']} | gated={exp['gated']} ---")

          # 🔴 config
          config = {
              "attn_type": attn["attn"],
              "pos_type": pos['pos'],
              "seq_len": ctx_len,
              "interleaved_conv" : exp['interleaved'],
              "gated_ffn" : exp['gated']
          }

          # 🔴 build fresh model
          model = build_model(config)

          optimizer = torch.optim.AdamW(
                            model.parameters(),
                            lr=5e-4,
                            betas=(0.9, 0.95),
                            weight_decay=0.1
                        )

          total_steps = len(train_loader) * epochs
          warmup_steps = int(0.1 * total_steps)

          scheduler = get_cosine_schedule_with_warmup(
              optimizer,
              num_warmup_steps=warmup_steps,
              num_training_steps=total_steps
          )

          loss_fn = torch.nn.CrossEntropyLoss(ignore_index = -100)

          # 🔴 train
          history = train(
              model,
              train_loader,
              val_loader,
              optimizer,
              scheduler,
              loss_fn,
              epochs=epochs  # keep small initially
          )

          # 🔴 store final metrics
          results = load_results()

          results.append({
              "attention": attn["attn"],
              "context": ctx_len,
              "pos" : pos["pos"],
              "train_loss": history["train_loss"][-1],
              "val_loss": history["val_loss"][-1],
              "ppl": history["val_perplexity"][-1],
              "throughput": history["throughput"][-1],
              "peak_mem": history["peak_memory"][-1],
              "epoch_time": history["epoch_time"][-1]
          })

          save_results(results)

          # 🔴 free memory before next run
          del model
          torch.cuda.empty_cache()

print("\nSaved results. Total runs:", len(results))

## For inference on diff lengths

In [ ]:
from transformers import get_cosine_schedule_with_warmup
import torch

torch.manual_seed(42)

epochs = 3

eval_lengths = [512, 1024, 2048]
base_tokens = 8192 # Keep batch sizes memory-friendly

for attn in attention_variants:
    for pos in pos_variants:
        print(f"\n=== Training: {attn['attn']} | {pos['pos']} ===")

        # 1. Setup and Train purely on 512
        config = {
            "attn_type": attn["attn"],
            "pos_type": pos["pos"],
            "seq_len": 512, # Training context
        }

        model = build_model(config).to(device)
        train_loader, val_loader_512, _ = get_dataloaders(seq_len=512, batch_size=base_tokens//512)

        optimizer = torch.optim.AdamW(
                          model.parameters(),
                          lr=5e-4,
                          betas=(0.9, 0.95),
                          weight_decay=0.1
                      )

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.1 * total_steps)

        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )

        loss_fn = torch.nn.CrossEntropyLoss(ignore_index = -100)

        # Train the model
        train(model, train_loader, val_loader_512, optimizer, scheduler, loss_fn, epochs=3)

        # 2. Extrapolation Evaluation Loop
        model.eval()
        for eval_len in eval_lengths:
            print(f"--- Evaluating on Context Length: {eval_len} ---")

            # Load the longer validation dataset
            _, val_loader, _ = get_dataloaders(seq_len=eval_len, batch_size=base_tokens//eval_len)

            total_val_loss = 0
            total_val_tokens = 0

            with torch.inference_mode():
                for batch in val_loader:

                    input_ids = batch["input_ids"].to(device)
                    labels = batch["labels"].to(device)
                    inputs = input_ids[:, :-1]
                    targets = labels[:, 1:]

                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        logits = model(inputs)
                        loss = loss_fn(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

                    batch_tokens = targets.numel()
                    total_val_loss += loss.item() * batch_tokens
                    total_val_tokens += batch_tokens

            avg_val_loss = total_val_loss / total_val_tokens
            val_ppl = math.exp(min(avg_val_loss, 25))

            print(f"Loss: {avg_val_loss:.4f} | Perplexity: {val_ppl:.2f}")

            results = load_results()

            # Save results
            results.append({
                "attention": attn["attn"],
                "pos": pos["pos"],
                "train_ctx": 512,
                "eval_ctx": eval_len,
                "val_loss": avg_val_loss,
                "ppl": val_ppl
            })

            save_results(results)

        # 🔴 free memory before next run
        del model
        torch.cuda.empty_cache()

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd


def plot_results(results, save_dir="figures"):

    # -----------------------------
    # Better plot styling
    # -----------------------------
    plt.rcParams.update({
        "figure.figsize": (7,5),
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "lines.linewidth": 2.5,
        "lines.markersize": 8
    })

    os.makedirs(save_dir, exist_ok=True)

    df = pd.DataFrame(results)
    df = df.sort_values(by=["context", "attention"])

    attn_types = df["attention"].unique()


    # -----------------------------
    # 1. Val Loss vs Context
    # -----------------------------
    plt.figure()

    for attn in attn_types:
        subset = df[df["attention"] == attn]
        plt.plot(
            subset["context"],
            subset["val_loss"],
            marker='o',
            label=attn
        )

    plt.xlabel("Context Length")
    plt.ylabel("Validation Loss")
    plt.title("Validation Loss vs Context Length")
    plt.legend()
    plt.grid(alpha=0.3)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/val_loss_vs_context.pdf", bbox_inches="tight")
    plt.show()


    # -----------------------------
    # 2. Throughput vs Context
    # -----------------------------
    plt.figure()

    for attn in attn_types:
        subset = df[df["attention"] == attn]
        plt.plot(
            subset["context"],
            subset["throughput"],
            marker='o',
            label=attn
        )

    plt.xlabel("Context Length")
    plt.ylabel("Tokens/sec")
    plt.title("Throughput vs Context Length")
    plt.legend()
    plt.grid(alpha=0.3)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/throughput_vs_context.pdf", bbox_inches="tight")
    plt.show()


    # -----------------------------
    # 3. Memory vs Context
    # -----------------------------
    plt.figure()

    for attn in attn_types:
        subset = df[df["attention"] == attn]
        plt.plot(
            subset["context"],
            subset["peak_mem"],
            marker='o',
            label=attn
        )

    plt.xlabel("Context Length")
    plt.ylabel("Peak Memory (GB)")
    plt.title("Memory Usage vs Context Length")
    plt.legend()
    plt.grid(alpha=0.3)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/memory_vs_context.pdf", bbox_inches="tight")
    plt.show()


    # -----------------------------
    # 4. Memory vs Loss Tradeoff
    # -----------------------------
    plt.figure()

    for attn in attn_types:
        subset = df[df["attention"] == attn]

        plt.scatter(
            subset["peak_mem"],
            subset["val_loss"],
            s=70,
            label=attn
        )

        for _, row in subset.iterrows():
            plt.text(
                row["peak_mem"],
                row["val_loss"],
                str(row["context"]),
                fontsize=9
            )

    plt.xlabel("Peak Memory (GB)")
    plt.ylabel("Validation Loss")
    plt.title("Memory vs Loss Tradeoff")
    plt.legend()
    plt.grid(alpha=0.3)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/memory_vs_loss_tradeoff.pdf", bbox_inches="tight")
    plt.show()


    # -----------------------------
    # 5. Speed vs Quality Tradeoff
    # -----------------------------
    plt.figure()

    for attn in attn_types:
        subset = df[df["attention"] == attn]

        plt.scatter(
            subset["throughput"],
            subset["val_loss"],
            s=70,
            label=attn
        )

        for _, row in subset.iterrows():
            plt.text(
                row["throughput"],
                row["val_loss"],
                str(row["context"]),
                fontsize=9
            )

    plt.xlabel("Throughput (Tokens/sec)")
    plt.ylabel("Validation Loss")
    plt.title("Speed vs Quality Tradeoff")
    plt.legend()
    plt.grid(alpha=0.3)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/speed_vs_quality_tradeoff.pdf", bbox_inches="tight")
    plt.show()

plot_results(results)

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd


def plot_positional_results(results, save_dir="pos_figures"):

    plt.rcParams.update({
        "figure.figsize": (7,5),
        "font.size": 12,
        "lines.linewidth": 2.5,
        "lines.markersize": 8
    })

    os.makedirs(save_dir, exist_ok=True)

    df = pd.DataFrame(results)

    # Only normal attention runs
    df = df[df["attention"] == "normal"]

    # Sort globally
    df = df.sort_values(by=["pos", "eval_ctx"])

    pos_types = df["pos"].unique()


    # ------------------------
    # Validation Loss Plot
    # ------------------------
    plt.figure()

    for pos in pos_types:

        subset = (
            df[df["pos"] == pos]
            .sort_values("eval_ctx")
        )

        plt.plot(
            subset["eval_ctx"],
            subset["val_loss"],
            marker="o",
            label=pos
        )

    plt.xlabel("Evaluation Context Length")
    plt.ylabel("Validation Loss")
    plt.title("Positional Encoding Extrapolation")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            save_dir,
            "positional_val_loss.pdf"
        ),
        bbox_inches="tight"
    )

    plt.show()


    # ------------------------
    # Perplexity Plot
    # ------------------------
    plt.figure()

    for pos in pos_types:

        subset = (
            df[df["pos"] == pos]
            .sort_values("eval_ctx")
        )

        plt.plot(
            subset["eval_ctx"],   # FIXED
            subset["ppl"],
            marker="o",
            label=pos
        )

    plt.xlabel("Evaluation Context Length")
    plt.ylabel("Perplexity")
    plt.title("Positional Encoding Extrapolation")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            save_dir,
            "positional_ppl.pdf"
        ),
        bbox_inches="tight"
    )

    plt.show()


# Run
plot_positional_results(results)

In [ ]:
import matplotlib.pyplot as plt

def plot_architecture_comparison(seq_lengths, metrics_dict, ylabel, title, save_filename=None):

    # Global plot styling
    plt.rcParams.update({
        "figure.figsize": (7, 5),
        "font.size": 12,
        "lines.linewidth": 2.5,
        "lines.markersize": 8
    })

    plt.figure()

    # Dynamically plot each architecture in the dictionary
    for label, values in metrics_dict.items():
        plt.plot(seq_lengths, values, marker='o', label=label)

    # Labeling and formatting
    plt.xlabel("Context Length")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    # Save if a filename is provided
    if save_filename:
        plt.savefig(save_filename, bbox_inches="tight")

    plt.show()